# 🛠️ Notebook 2: Airline Management — Implementation

We implement the clean design from Notebook 1, one concept at a time:

1. Core types (Passenger, Seat, Aircraft, Flight).
2. Booking + pricing table.
3. Cancellation with a refund policy.
4. Search across many flights.
5. Crew assignment.
6. Concurrency: a realistic race condition and how to fix it.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/airline-management
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. Core types

- `Passenger` and `Seat` are **immutable value objects** (`frozen=True`): two seats with the same number are "the same seat."
- `SeatClass` is an **enum** — fixed set of choices, no typos.
- `PRICING` is a tiny **lookup table**. Adding Premium Economy = one new line. No subclassing.


In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
from itertools import count
from threading import Lock

class SeatClass(Enum):
    ECONOMY = "ECON"
    BUSINESS = "BIZ"
    FIRST = "FIRST"

# Data-shaped variation → lookup table, not subclasses.
PRICING = {
    SeatClass.ECONOMY:  200,
    SeatClass.BUSINESS: 700,
    SeatClass.FIRST:   1800,
}

@dataclass(frozen=True)
class Passenger:
    id: int
    name: str
    passport: str

@dataclass(frozen=True)
class Seat:
    number: str               # e.g. "12A"
    seat_class: SeatClass

@dataclass
class Aircraft:
    model: str                # e.g. "A320"
    seats: list[Seat]

# Crew are NOT passengers: different role, different lifecycle, no booking.
# They live here with the other core value objects because `Flight` needs
# them to enforce its "a flight cannot fly without a pilot" invariant.
class CrewRole(Enum):
    PILOT = "PILOT"
    ATTENDANT = "ATTENDANT"

@dataclass(frozen=True)
class CrewMember:
    id: int
    name: str
    role: CrewRole

# Quick sanity check
demo_seats = [Seat("1A", SeatClass.FIRST), Seat("5A", SeatClass.ECONOMY)]
print(Aircraft("A320", demo_seats))


## 2. `Flight` + `Booking` (+ two design decisions)

Notice how `Flight` owns only **availability** — the seat *map* lives on `Aircraft`. That way the same A320 can fly UA100 today and UA250 tomorrow without duplicating seats.

`Booking` is its own object. It has a `status` so we can cancel it without losing the audit trail.

Two things in the cell below are worth slowing down for:

| Decision | Why |
|---|---|
| `RefundPolicy` is an injected **Strategy** | Notebook 1 complained that `BadFlight` hard-coded the price inside `book`. A hard-coded *refund* table is the same smell. Pull the rule behind an interface → `NonRefundablePolicy` is a new class, not an edit to `Flight`. |
| `Booking.mark_cancelled()` instead of `booking.status = ...` | "You cannot cancel twice" is a `Booking` invariant, so it belongs on `Booking`. If any caller can assign to `status`, the rule is unenforceable. |
| `crew` is a real field with `assign_crew()` | Attributes bolted on from outside (`flight.crew = [...]`) are invisible to readers, to `repr`, and to type checkers — and no invariant can guard them. |


In [ ]:
_bids = count(1)   # simple booking id generator

class BookingStatus(Enum):
    CONFIRMED = "CONFIRMED"
    CANCELLED = "CANCELLED"


# ── Refund policy: a Strategy ─────────────────────────────────────────────
# Notebook 1 criticised `BadFlight` for hard-coding the price inside `book`.
# The same argument applies to the *refund* rule: airlines change it per fare
# class, per route, per promotion. So we pull it behind an interface and
# inject it. Adding a new rule = one new class; `Flight` never changes.
class RefundPolicy(ABC):
    @abstractmethod
    def refund_fraction(self, lead: timedelta) -> float:
        """How much of the fare to return, given the time left before departure."""


class StandardRefundPolicy(RefundPolicy):
    """>= 7 days out → 100%; >= 1 day out → 50%; otherwise → nothing."""
    def refund_fraction(self, lead: timedelta) -> float:
        if lead >= timedelta(days=7):
            return 1.0
        if lead >= timedelta(days=1):
            return 0.5
        return 0.0


class NonRefundablePolicy(RefundPolicy):
    """Basic-economy style fare: you may cancel, but you get nothing back."""
    def refund_fraction(self, lead: timedelta) -> float:
        return 0.0


@dataclass
class Booking:
    passenger: Passenger
    flight: "Flight"
    seat: Seat
    price: float
    status: BookingStatus = BookingStatus.CONFIRMED
    id: int = field(default_factory=lambda: next(_bids))

    def ticket(self) -> str:
        return (f"TKT#{self.id}  {self.flight.number}  "
                f"{self.flight.origin}→{self.flight.destination}  "
                f"seat {self.seat.number} ({self.seat.seat_class.value})  "
                f"${self.price}  [{self.status.value}]")

    def mark_cancelled(self) -> None:
        """A Booking guards its own status — nobody else may assign to it.

        `Flight.cancel` used to do `booking.status = CANCELLED` directly. That
        works, but it puts one object's invariant ("you cannot cancel twice")
        in *another* object's method. Encapsulation means the rule lives with
        the data it protects.
        """
        if self.status is BookingStatus.CANCELLED:
            raise ValueError("already cancelled")
        self.status = BookingStatus.CANCELLED


@dataclass
class Flight:
    number: str
    origin: str
    destination: str
    departs: datetime
    aircraft: Aircraft
    # Injected dependency → swap the rule without touching Flight (Open/Closed).
    refund_policy: RefundPolicy = field(default_factory=StandardRefundPolicy)
    # Declared as a real field, not bolted on later with `flight.crew = [...]`.
    crew: list[CrewMember] = field(default_factory=list)
    _available: dict[str, bool] = field(init=False)
    _lock: Lock = field(init=False, repr=False)

    def __post_init__(self):
        self._available = {s.number: True for s in self.aircraft.seats}
        self._lock = Lock()     # protects concurrent bookings (see section 6)

    def available_seats(self, seat_class: SeatClass | None = None) -> list[Seat]:
        return [s for s in self.aircraft.seats
                if self._available[s.number]
                and (seat_class is None or s.seat_class == seat_class)]

    def book(self, passenger: Passenger, seat: Seat) -> Booking:
        with self._lock:
            if not self._available.get(seat.number, False):
                raise ValueError(f"seat {seat.number} not available")
            self._available[seat.number] = False
        return Booking(passenger, self, seat, PRICING[seat.seat_class])

    def cancel(self, booking: Booking, now: datetime | None = None) -> float:
        """Cancel a booking and return the refunded amount.

        Flight owns *seat release*; the policy owns *how much money comes
        back*; the Booking owns *its own status*. Three responsibilities,
        three objects.
        """
        if booking.flight is not self:
            raise ValueError("booking is not on this flight")
        booking.mark_cancelled()          # raises if already cancelled
        now = now or datetime.now()
        pct = self.refund_policy.refund_fraction(self.departs - now)
        with self._lock:
            self._available[booking.seat.number] = True
        return round(booking.price * pct, 2)

    def assign_crew(self, members: list[CrewMember]) -> None:
        """Enforce a real business invariant: no pilot, no flight.

        A dataclass field alone cannot stop `flight.crew.append(...)` from
        producing a pilotless flight, so the *mutation* goes through a method
        that validates. That is what "invariant enforcement" means in practice.
        """
        if not any(m.role is CrewRole.PILOT for m in members):
            raise ValueError("a flight needs at least one pilot")
        if len({m.id for m in members}) != len(members):
            raise ValueError("duplicate crew member on the same flight")
        self.crew = list(members)          # copy: callers cannot mutate ours


### Try it: book, ticket, double-book, cancel

In [ ]:
# Tiny A320-ish seat map: 4 economy + 2 business + 1 first
seats = (
    [Seat(f"{r}{c}", SeatClass.ECONOMY)  for r in (5, 6) for c in "AB"]
    + [Seat(f"{r}{c}", SeatClass.BUSINESS) for r in (2,)  for c in "AB"]
    + [Seat("1A", SeatClass.FIRST)]
)
plane = Aircraft("A320", seats)

fl = Flight("UA100", "SFO", "JFK",
            datetime(2025, 6, 1, 9, 0), plane)

alice = Passenger(1, "Alice", "P111")
bob   = Passenger(2, "Bob",   "P222")

print("Business seats available:", fl.available_seats(SeatClass.BUSINESS))

b1 = fl.book(alice, fl.available_seats(SeatClass.BUSINESS)[0])
b2 = fl.book(bob,   fl.available_seats(SeatClass.ECONOMY)[0])
print(b1.ticket())
print(b2.ticket())

# Double-booking the same seat must fail
try:
    fl.book(bob, b1.seat)
except ValueError as e:
    print("expected error:", e)

# Cancel Alice's booking 10 days before departure → full refund
refund = fl.cancel(b1, now=fl.departs - timedelta(days=10))
print(f"Alice refund: ${refund}  |  seat free again? "
      f"{b1.seat in fl.available_seats()}")

# Cancelling twice must fail — the invariant lives on Booking itself.
try:
    fl.cancel(b1, now=fl.departs - timedelta(days=10))
except ValueError as e:
    print("expected error:", e)

# Same flight, same timing, DIFFERENT policy → different money. Flight code
# did not change at all; we only injected another strategy object.
budget = Flight("UA101", "SFO", "JFK", datetime(2025, 6, 1, 9, 0), plane,
                refund_policy=NonRefundablePolicy())
b3 = budget.book(alice, budget.available_seats(SeatClass.ECONOMY)[0])
print("basic-economy refund 10 days out: $"
      f"{budget.cancel(b3, now=budget.departs - timedelta(days=10))}")


## 3. Searching flights

`FlightSearch` is a separate service — it *queries* flights; it doesn't *own* them. Putting search inside `Flight` would violate single responsibility and make it hard to add filters (airline, stops, price…).


In [ ]:
@dataclass
class FlightSearch:
    flights: list[Flight]

    def search(self, origin: str, destination: str, date) -> list[Flight]:
        return [f for f in self.flights
                if f.origin == origin
                and f.destination == destination
                and f.departs.date() == date]

# Build a small schedule: 3 flights on different days / routes
plane2 = Aircraft("A320", [Seat("5A", SeatClass.ECONOMY)])
schedule = [
    fl,
    Flight("UA200", "SFO", "JFK", datetime(2025, 6, 1, 18, 0), plane2),
    Flight("UA300", "SFO", "LAX", datetime(2025, 6, 1, 8, 0),  plane2),
]
search = FlightSearch(schedule)
results = search.search("SFO", "JFK", datetime(2025, 6, 1).date())
print("Found", len(results), "SFO→JFK flights on 2025-06-01:")
for f in results:
    print(" ", f.number, f.departs.strftime("%H:%M"))


## 4. Crew — and enforcing an invariant

Pilots and attendants are *not* passengers — different role, different lifecycle. They were declared next to the other value objects in section 1 because `Flight` needs them.

The interesting part is `assign_crew`. A plain field cannot stop somebody building a **pilotless flight**; a method can. Whenever a rule must always hold, route the mutation through a method that checks it.

In [ ]:
# Crew goes through assign_crew(), which validates the invariant.
fl.assign_crew([
    CrewMember(101, "Capt. Smith", CrewRole.PILOT),
    CrewMember(102, "F.O. Jones",  CrewRole.PILOT),
    CrewMember(201, "Maya",        CrewRole.ATTENDANT),
])
print(f"{fl.number} crew:")
for m in fl.crew:
    print(" -", m.role.value, m.name)

# Invariant 1: a flight cannot fly without a pilot.
try:
    fl.assign_crew([CrewMember(202, "Sam", CrewRole.ATTENDANT)])
except ValueError as e:
    print("expected error:", e)

# Invariant 2: the same person cannot hold two seats on one flight.
try:
    fl.assign_crew([CrewMember(101, "Capt. Smith", CrewRole.PILOT),
                    CrewMember(101, "Capt. Smith", CrewRole.PILOT)])
except ValueError as e:
    print("expected error:", e)

print("crew survived the bad assignments:", [m.name for m in fl.crew])

## 5. Concurrency — the realistic bug

Two customers on two browser tabs click "Book seat 2A" at almost the same moment.

Without a lock, *both* checks see the seat as free and *both* succeed. That's a real production bug in booking systems.

Our `Flight.book` already uses a `Lock` — let's prove it matters by forcing the race with many threads.


In [ ]:
from threading import Thread

# Fresh flight with ONE business seat — the contested one.
tiny = Aircraft("A320", [Seat("2A", SeatClass.BUSINESS)])
hot_flight = Flight("UA999", "SFO", "JFK", datetime(2025, 6, 1, 9, 0), tiny)

winners, losers = [], []

def try_book(name):
    p = Passenger(hash(name) & 0xFFFF, name, "PX")
    try:
        b = hot_flight.book(p, Seat("2A", SeatClass.BUSINESS))
        winners.append(b)
    except ValueError:
        losers.append(name)

threads = [Thread(target=try_book, args=(f"user{i}",)) for i in range(50)]
for t in threads: t.start()
for t in threads: t.join()

print(f"winners: {len(winners)}  losers: {len(losers)}")
assert len(winners) == 1, "lock is broken — two people got the same seat!"
print("✅ Exactly one booking succeeded, as expected.")


### What would be *wrong* without the lock?

If we replaced `Flight.book` with:

```python
def book(self, passenger, seat):
    if not self._available[seat.number]:   # check
        raise ValueError("not available")
    self._available[seat.number] = False   # mutate
    return Booking(...)
```

Between *check* and *mutate*, another thread can sneak in — both threads see `True`, both set `False`, both return bookings. This is called a **check-then-act race**. Always guard check+mutate pairs with a lock (or use an atomic operation / a DB transaction in production).


## 6. Verify the design

Prose claims are cheap. These assertions are the design written down as
executable rules — if a later refactor breaks one, the cell fails loudly.

In [ ]:
def fresh_flight(**kw):
    seats = [Seat("1A", SeatClass.FIRST),
             Seat("2A", SeatClass.BUSINESS),
             Seat("5A", SeatClass.ECONOMY)]
    return Flight("UA777", "SFO", "JFK", datetime(2025, 6, 1, 9, 0),
                  Aircraft("A320", seats), **kw)

pax = Passenger(9, "Test", "P999")

# --- Aircraft owns the seat map; Flight owns availability ----------------
plane_shared = Aircraft("A320", [Seat("1A", SeatClass.FIRST)])
f1 = Flight("UA1", "SFO", "JFK", datetime(2025, 6, 1), plane_shared)
f2 = Flight("UA2", "JFK", "SFO", datetime(2025, 6, 2), plane_shared)
f1.book(pax, plane_shared.seats[0])
assert f2.available_seats() == plane_shared.seats, \
    "booking one flight must not consume the seat on another flight of the same plane"

# --- a seat can be sold exactly once ------------------------------------
f = fresh_flight()
biz = f.available_seats(SeatClass.BUSINESS)[0]
f.book(pax, biz)
try:
    f.book(pax, biz); raise AssertionError("double booking was allowed")
except ValueError:
    pass
assert biz not in f.available_seats()

# --- pricing is data, not behaviour: one table lookup, no subclasses -----
assert set(PRICING) == set(SeatClass), "every seat class needs a price row"
assert f.book(pax, f.available_seats(SeatClass.FIRST)[0]).price == PRICING[SeatClass.FIRST]

# --- Booking guards its own status --------------------------------------
f = fresh_flight()
b = f.book(pax, f.available_seats(SeatClass.ECONOMY)[0])
assert b.status is BookingStatus.CONFIRMED
f.cancel(b, now=f.departs - timedelta(days=10))
assert b.status is BookingStatus.CANCELLED
try:
    b.mark_cancelled(); raise AssertionError("double cancel was allowed")
except ValueError:
    pass

# --- the refund Strategy is genuinely swappable -------------------------
lead10 = timedelta(days=10)
assert StandardRefundPolicy().refund_fraction(lead10) == 1.0
assert StandardRefundPolicy().refund_fraction(timedelta(days=3)) == 0.5
assert StandardRefundPolicy().refund_fraction(timedelta(hours=2)) == 0.0
assert NonRefundablePolicy().refund_fraction(lead10) == 0.0
cheap = fresh_flight(refund_policy=NonRefundablePolicy())
b = cheap.book(pax, cheap.available_seats(SeatClass.FIRST)[0])
assert cheap.cancel(b, now=cheap.departs - lead10) == 0.0

# --- cancelling releases the seat ---------------------------------------
assert cheap.available_seats(SeatClass.FIRST), "cancelled seat must be resellable"

# --- crew invariants -----------------------------------------------------
f = fresh_flight()
for bad in ([], [CrewMember(1, "A", CrewRole.ATTENDANT)],
            [CrewMember(1, "A", CrewRole.PILOT), CrewMember(1, "A", CrewRole.PILOT)]):
    try:
        f.assign_crew(bad); raise AssertionError(f"accepted invalid crew: {bad}")
    except ValueError:
        pass
assert f.crew == [], "a rejected assignment must leave the flight untouched"
f.assign_crew([CrewMember(1, "A", CrewRole.PILOT)])
assert len(f.crew) == 1

# --- a booking belongs to exactly one flight ----------------------------
other = fresh_flight()
b = other.book(pax, other.available_seats(SeatClass.ECONOMY)[0])
try:
    f.cancel(b); raise AssertionError("cancelled a booking on the wrong flight")
except ValueError:
    pass

print("✅ all design invariants hold")

## 🎯 Your turn — small, high-value extensions

- **Overbooking**: allow up to 5% more bookings than seats; assign overflow at check-in.
- **Seat holds**: `hold(seat, ttl=10min)` prevents others from booking while the passenger enters payment. Release on expiry.
- **Loyalty discount**: wrap `PRICING` in a function `price_for(passenger, seat_class)` — no new classes needed.
- **Multi-leg itineraries**: a `Trip` that owns many `Booking`s on connecting `Flight`s, with a combined price.
- **Persistence**: serialize flights/bookings with `dataclasses.asdict`, reload on restart.


## 🧠 Takeaways

1. **Start from requirements** — not from classes. Classes fall out of responsibilities.
2. **Subclass for behaviour, not for data.** Seat classes vary in *price*, not in *what a seat does* → enum + lookup table. Refund rules vary in *behaviour* → Strategy objects. Same question, opposite answers, and that is the whole skill.
3. **Separate concerns**: seat map (`Aircraft`) vs availability (`Flight`) vs transaction (`Booking`) vs query (`FlightSearch`) vs money-back rule (`RefundPolicy`).
4. **Put each invariant on the object that owns the data.** "Cannot cancel twice" lives on `Booking`; "cannot fly without a pilot" lives on `Flight.assign_crew`. A rule enforced by convention is not enforced.
5. **Never bolt attributes on from outside.** `flight.crew = [...]` silently creates a second, unvalidated version of your model.
6. **Concurrency matters**: check-then-act without a lock is a bug. A `Lock` around the critical section is enough for a single process; in production use your DB transaction.